# Auto-Commit vs Manual Commit — explained for a total Kafka beginner

Notebook 6 gave us this consumer flow:

```text
subscribe → poll → decode → process → commit → repeat → close
```

This notebook asks one question:

> **Who moves the consumer group's saved bookmark, and exactly when does it move?**

That is all “auto-commit versus manual commit” means.

## What this notebook deliberately avoids

- No unit tests or fake Kafka clients.
- No reusable consumer framework or metrics class.
- No database implementation.
- No batching, worker pools, or rebalance callback code.
- No attempt to imitate a process crash by killing Jupyter.

Those topics matter later. Here we make commit timing understandable first.

## Complete learning flow

```text
Stage 1: separate four moments
         fetched → processed → stored locally → committed to Kafka
                                  |
                                  v
Stage 2: compare crash timing
         commit early risks skipped work
         commit late risks repeated work
                                  |
                                  v
Stage 3: compare three configurations
         automatic / manual / automatic with manual storage
                                  |
                                  v
Stage 4: build one clear manual-commit loop
                                  |
                                  v
Stage 5: run an optional live example
                                  |
                                  v
Stage 6: understand the production limits
```


## Before code: one-minute recap

| Word | Plain-English meaning |
|---|---|
| **Record offset** | The record's numbered position inside one partition. |
| **Committed offset** | The next position a consumer group should read from. |
| **Consumer group** | A named team that shares partition work and owns saved progress. |
| **Commit** | Save that group's next reading position in Kafka. |
| **Replay / redelivery** | Kafka gives the application a record it previously received. |
| **Crash window** | A small period between two actions where a crash changes the result. |

If the application successfully processes record offset `7`, its next reading position is `8`:

```text
Partition 0:  [5] [6] [7] [8] [9]
                       ^   ^
                       |   +-- committed next position
                       +------ processed record
```

The committed number means “read from here next,” not “this was the last record processed.”


---
# Stage 0.0 — Prepare the notebook

Install the Python client once:

```bash
python -m pip install "confluent-kafka>=2.6"
```

Start this repository's learning broker from the repository root:

```bash
docker compose -f notebook/lab/docker-compose.yml up -d
```

The learning broker is available at `localhost:9092`. Keep the live switch off while reading. Run the notebook from top to bottom so every later cell receives the names created earlier.


In [1]:
# Stage 0.1 — Import only what this lesson uses.
from __future__ import annotations

# Standard-library imports convert JSON and read environment settings.
import json
import os
from typing import Any

# Kafka is optional while reading, so missing installation does not break the lesson.
try:
    from confluent_kafka import Consumer, KafkaException, Message, Producer

    KAFKA_LIBRARY_AVAILABLE = True
except ImportError:
    Consumer = KafkaException = Message = Producer = None
    KAFKA_LIBRARY_AVAILABLE = False


### Code explanation — every meaningful line in Stage 0.1

Blank lines and closing brackets only format the code; they do not perform an action.

| Code | Meaning |
|---|---|
| The Stage 0.1 comment | Marks this code's position in the notebook flow. |
| `from __future__ import annotations` | Delays evaluation of type hints. Functions can still be defined while the optional Kafka package is absent. |
| `import json` | Loads Python's JSON converter for dictionary ↔ text conversion. |
| `import os` | Loads environment-variable support so another environment can change the broker address without editing code. |
| `from typing import Any` | Imports a type used only at mixed-value JSON and third-party configuration boundaries. Validation restores a known shape before business work. |
| `try:` | Starts an import attempt that is allowed to fail safely. |
| `from confluent_kafka import ...` | Imports the real consumer, Kafka exception, message type, and the tiny producer used to seed the optional live lab. |
| `KAFKA_LIBRARY_AVAILABLE = True` | Records that the Kafka import succeeded. |
| `except ImportError:` | Selects the fallback path when the package is not installed. |
| `Consumer = ... = None` | Creates the expected names without pretending that real Kafka classes exist. Live functions check them before use. |
| `KAFKA_LIBRARY_AVAILABLE = False` | Records that network-connected Kafka code must not run yet. |


In [2]:
# Stage 0.2 — Keep environment and lab settings visible.
BOOTSTRAP_SERVERS: str = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
OFFSET_LAB_TOPIC: str = "beginner_offset_commit_lab"
MANUAL_GROUP_ID: str = "beginner-offset-manual-v1"
RUN_LIVE_KAFKA: bool = True

print("Kafka library installed:", KAFKA_LIBRARY_AVAILABLE)
print("Broker address:", BOOTSTRAP_SERVERS)
print("Lab topic:", OFFSET_LAB_TOPIC)
print("Manual consumer group:", MANUAL_GROUP_ID)
print("Live Kafka enabled:", RUN_LIVE_KAFKA)


Kafka library installed: True
Broker address: localhost:9092
Lab topic: beginner_offset_commit_lab
Manual consumer group: beginner-offset-manual-v1
Live Kafka enabled: True


### Code explanation — every meaningful line in Stage 0.2

| Code | Meaning |
|---|---|
| `BOOTSTRAP_SERVERS = os.getenv(...)` | Uses the environment value when present; otherwise uses the local learning broker. “Bootstrap” means the first address used to discover Kafka. |
| `OFFSET_LAB_TOPIC = ...` | Names the dedicated topic used only by this offset lesson. |
| `MANUAL_GROUP_ID = ...` | Names the group whose committed position the manual consumer will use. Keeping the name stable lets the group resume. |
| `RUN_LIVE_KAFKA = False` | Prevents accidental network activity. Change it only after the broker is ready. |
| Each `print(...)` | Shows the exact setting later cells will use, which removes hidden configuration surprises. |


---
# Stage 1.0 — Separate four different moments

People often say “the consumer got offset 7” without saying which kind of progress they mean. Keep these moments separate:

| Moment | What actually happened? | Has Kafka's group bookmark moved? |
|---|---|---|
| **1. Fetched** | `poll()` returned record `7` to the application. | Not necessarily. |
| **2. Processed** | The application's business handler finished record `7`. | Not necessarily. |
| **3. Stored locally** | The client marked next position `8` as eligible for a later commit. | No. It is still client-side state. |
| **4. Committed** | Kafka saved position `8` for this group and partition. | Yes. |

The difficult-looking phrase **offset management** only means deciding when stages 3 and 4 happen.

## Desk-bookmark analogy

```text
Kafka record 7 arrives on your desk        = fetched
You finish its business work               = processed
You write “next page: 8” on a sticky note  = stored locally
The library files that note with Kafka     = committed
```

A sticky note still on your desk can disappear in a crash. Only the broker-side committed position survives for the consumer group.


In [3]:
def next_reading_position(processed_offset: int) -> int:
    """Convert a processed record offset into the next position to read.

    Args:
        processed_offset: Non-negative offset of one completed record.

    Returns:
        The following offset, which is the consumer group's resume position.

    Raises:
        ValueError: The teaching example receives a negative record offset.

    Production tradeoff:
        Real clients calculate this for ``commit(message=...)``. This helper exists
        only to make the offset-plus-one convention visible to a beginner.
    """
    # Stage 1.1 — Reject an impossible value for this simple example.
    if processed_offset < 0:
        raise ValueError("The record offset must be zero or greater")

    # Stage 1.2 — A commit stores where the group should read next.
    return processed_offset + 1


for example_offset in (0, 7, 41):
    next_position = next_reading_position(example_offset)
    print(f"Processed offset {example_offset}; next reading position {next_position}")


Processed offset 0; next reading position 1
Processed offset 7; next reading position 8
Processed offset 41; next reading position 42


### Code explanation — every meaningful line in Stage 1

| Code | Meaning |
|---|---|
| `def next_reading_position(...) -> int:` | Defines a function accepting one integer and returning one integer. Defining it does not call it. |
| The function docstring | Documents the input, output, possible error, and why this teaching helper exists. |
| `if processed_offset < 0:` | Checks the simple exercise's rule that stored record offsets are non-negative. |
| `raise ValueError(...)` | Stops the function with a direct explanation when the input is invalid. |
| `return processed_offset + 1` | Produces the position immediately after the completed record. |
| `for example_offset in (0, 7, 41):` | Repeats the indented lines once for each example offset. Parentheses create a fixed tuple of examples. |
| `next_position = ...` | Calls the function and saves its answer. |
| `print(f"...")` | Displays each record offset and its next position. The `f` inserts the two variables into the text. |


---
# Stage 2.0 — Commit timing changes the failure result

There are two fundamental orders.

## Stage 2.1 — Process first, commit second

```text
POLL record 7
      |
      v
PROCESS record 7 successfully
      |
      |  <--- crash here
      v
COMMIT next position 8
```

If the application crashes in the marked window, Kafka still has the older bookmark. Record `7` may be delivered again after restart.

That is **replay risk**: work may repeat, so the business action should be idempotent. Idempotent means that receiving the same event again does not produce an unwanted second result.

This ordering is the normal foundation for **at-least-once processing attempts**.

## Stage 2.2 — Commit first, process second

```text
POLL record 7
      |
      v
COMMIT next position 8
      |
      |  <--- crash here
      v
PROCESS record 7
```

If the application crashes in the marked window, Kafka's bookmark already says `8`. The restarted group can continue after unfinished record `7`.

That is **loss risk**: business work can be skipped. This ordering is associated with at-most-once attempts when the commit is confirmed before processing.

## The choice in one table

| Ordering | What it protects against | What it risks |
|---|---|---|
| Process → commit | Skipping work because progress moved too soon | Repeating work after a crash |
| Commit → process | Repeating work after a confirmed early commit | Skipping unfinished work |

For important order processing, this notebook chooses **process first, commit second** and requires duplicate-safe business behavior.


## Stage 2.3 — A commit is not a business transaction

Suppose the handler writes an order to PostgreSQL and then commits Kafka offset `8`.

These are two separate systems:

```text
PostgreSQL transaction succeeds
             |
             | application can crash here
             v
Kafka offset commit succeeds
```

Kafka cannot infer whether PostgreSQL succeeded. A Kafka offset commit means only:

> “This consumer group should resume at this topic-partition position.”

It does not mean:

- the producer received an acknowledgement;
- the database transaction exists;
- an email was sent;
- a payment was charged exactly once.

If database success happens before a crash and the Kafka commit does not, the record can return. A production handler typically uses the event's stable `event_id` as an idempotency key so the repeated attempt does not repeat the business effect.


---
# Stage 3.0 — Compare the three configurations

Two independent switches control the behavior:

- `enable.auto.offset.store`: should the client automatically make a fetched position eligible for commit?
- `enable.auto.commit`: should a background client process periodically send eligible positions to Kafka?

“Store” and “commit” are not synonyms:

```text
store offset locally in client memory
                 |
                 v
commit offset to Kafka for the consumer group
```


In [4]:
# Stage 3.1 — Mode A: the client stores and commits automatically.
# Result: Kafka progress can move before the business handler finishes.
AUTOMATIC_CONFIG: dict[str, Any] = {
    # Address used to make the initial connection to the Kafka cluster.
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    # Separate group name: Kafka stores this mode's progress independently.
    "group.id": "beginner-offset-automatic-v1",
    # With no valid commit, begin at the oldest record Kafka still retains.
    "auto.offset.reset": "earliest",
    # True: poll automatically makes the fetched record's next offset eligible.
    "enable.auto.offset.store": True,
    # True: the client periodically sends eligible offsets to Kafka.
    "enable.auto.commit": True,
    # Try an automatic broker commit every 5,000 ms (five seconds).
    "auto.commit.interval.ms": 5000,
}

# Stage 3.2 — Mode B: application code commits explicitly after success.
# Result: only our later commit(message=...) call moves Kafka's bookmark.
MANUAL_COMMIT_CONFIG: dict[str, Any] = {
    # Use the same Kafka cluster as the other modes.
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    # This stable group owns the manual consumer's independent progress.
    "group.id": MANUAL_GROUP_ID,
    # With no valid commit, begin at the oldest retained record.
    "auto.offset.reset": "earliest",
    # False: fetching a record must not automatically mark it as completed.
    "enable.auto.offset.store": False,
    # False: no background task is allowed to commit progress for us.
    "enable.auto.commit": False,
}

# Stage 3.3 — Mode C: application stores after success; client commits later.
# Result: our code chooses what is safe, but the client chooses commit timing.
MANUAL_STORE_CONFIG: dict[str, Any] = {
    # Use the same Kafka cluster as Modes A and B.
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    # Separate group name: this mode gets its own saved Kafka bookmark.
    "group.id": "beginner-offset-manual-store-v1",
    # With no valid commit, begin at the oldest retained record.
    "auto.offset.reset": "earliest",
    # False: our code calls store_offsets() only after processing succeeds.
    "enable.auto.offset.store": False,
    # True: the client later commits the positions our code stored locally.
    "enable.auto.commit": True,
    # Try to send those stored positions to Kafka every five seconds.
    "auto.commit.interval.ms": 5000,
}

print("Three offset configurations are ready.")


Three offset configurations are ready.


### Code explanation — every meaningful line in the three configurations

The dictionary values mix strings, booleans, and integers, so `Any` is deliberately limited to this third-party configuration boundary.

| Code | Meaning |
|---|---|
| `AUTOMATIC_CONFIG = {` | Begins Mode A, where the client controls both local storage and periodic broker commits. |
| `"bootstrap.servers": ...` | Tells the client how to find Kafka. It does not control commit timing. |
| `"group.id": ...` | Gives this mode its own saved group progress so it does not share the manual mode's bookmark. |
| `"auto.offset.reset": "earliest"` | Uses the oldest retained record only if this group has no valid committed position. It does not overwrite an existing commit. |
| `"enable.auto.offset.store": True` | Allows a fetched/delivered record position to become eligible automatically, before the application necessarily finishes processing. |
| `"enable.auto.commit": True` | Enables periodic background commits of eligible/stored positions. |
| `"auto.commit.interval.ms": 5000` | Requests a background commit interval of 5,000 milliseconds, or five seconds. This is timing, not a promise that business processing finished. |
| `MANUAL_COMMIT_CONFIG = {` | Begins Mode B, the clearest correctness-first teaching configuration. |
| Mode B's two `False` values | Stop both automatic local storage and automatic broker commits. Application code explicitly commits a particular successfully processed message. |
| `MANUAL_STORE_CONFIG = {` | Begins Mode C, the middle ground. |
| Mode C's store `False` | Application code decides when processing made an offset eligible. |
| Mode C's commit `True` | The client periodically sends eligible positions to Kafka. |
| The final `print(...)` | Confirms that all three dictionaries were created; it does not connect to Kafka. |


## Stage 3.4 — Why the fully automatic mode can skip work

A possible timeline is:

```text
TIME       KAFKA CLIENT                         BUSINESS HANDLER

t0         poll returns record 7
t1         automatically stores position 8
t2                                              starts record 7
t3         background commits position 8
t4                                              application crashes
t5         restarted group resumes at 8         record 7 was unfinished
```

This does not claim every automatic-commit crash loses work. If the background commit has not happened yet, record `7` may replay instead. The problem is that the saved bookmark timing is not tied directly to business success.

Also remember: `auto.offset.reset="earliest"` does not rescue record `7` when a valid committed position `8` already exists. Offset reset is only a fallback when there is no usable committed position.


---
# Stage 4.0 — Build the clear manual-commit baseline

The code will follow one visible contract:

```text
poll normal message
        |
        v
decode and validate
        |
        v
business handler succeeds
        |
        v
synchronous commit succeeds
        |
        v
count the record as complete
```

If decoding, handling, or committing fails, the function stops and closes the consumer. It does not pretend that progress is safe.


In [5]:
# Stage 4.1 — Create one event and the exact bytes used by the lab.
SAMPLE_ORDER_EVENT: dict[str, Any] = {
    "event_id": "evt-offset-1001",
    "event_type": "order.created",
    "schema_version": 1,
    "data": {
        "order_id": "order-1001",
        "amount": 42.50,
    },
}

SAMPLE_ORDER_BYTES: bytes = json.dumps(SAMPLE_ORDER_EVENT).encode("utf-8")

print("Application event:", SAMPLE_ORDER_EVENT)
print("Kafka value:", SAMPLE_ORDER_BYTES)


Application event: {'event_id': 'evt-offset-1001', 'event_type': 'order.created', 'schema_version': 1, 'data': {'order_id': 'order-1001', 'amount': 42.5}}
Kafka value: b'{"event_id": "evt-offset-1001", "event_type": "order.created", "schema_version": 1, "data": {"order_id": "order-1001", "amount": 42.5}}'


### Code explanation — every meaningful line in Stage 4.1

| Code | Meaning |
|---|---|
| `SAMPLE_ORDER_EVENT = {` | Creates one predictable Python dictionary for the whole lesson. |
| `event_id` | Stable event identity. A production handler can use it to recognize a replay. |
| `event_type` | Describes what happened. |
| `schema_version` | Identifies the event shape understood by this code. |
| `data = { ... }` | Holds the order-specific fields: identity and amount. |
| `json.dumps(...)` | Serializes the dictionary into JSON text. |
| `.encode("utf-8")` | Converts the text to bytes, which are appropriate for a Kafka value. |
| The two `print(...)` calls | Display the application form and the byte form. They do not send anything. |


In [6]:
class InvalidOrderEvent(ValueError):
    """Report that record bytes cannot become this lesson's order event.

    Input boundary: null, non-JSON, or unsupported event data.
    Output contract: no partially validated dictionary escapes the decoder.
    Failure behavior: the polling loop stops before business work and commit.
    """


def decode_order_event(raw_value: bytes | None) -> dict[str, Any]:
    """Decode and validate one order-event value from Kafka.

    Args:
        raw_value: Bytes returned by ``message.value()``, or ``None``.

    Returns:
        One supported ``order.created`` event dictionary.

    Raises:
        InvalidOrderEvent: Bytes are missing, malformed, or unsupported.

    Production tradeoff:
        Stopping avoids silently skipping unknown data, but a permanently bad
        record requires an explicit retry or quarantine policy.
    """
    # Stage 4.2 — Reject a missing Kafka value.
    if raw_value is None:
        raise InvalidOrderEvent("The Kafka record has no value")

    # Stage 4.3 — Convert bytes to text, then parse the JSON text.
    try:
        event = json.loads(raw_value.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError) as error:
        raise InvalidOrderEvent("The value must be valid UTF-8 JSON") from error

    # Stage 4.4 — Require the event shape needed by the handler.
    if not isinstance(event, dict):
        raise InvalidOrderEvent("The JSON value must be an object")
    if event.get("event_type") != "order.created":
        raise InvalidOrderEvent("Only event_type='order.created' is supported")
    if event.get("schema_version") != 1:
        raise InvalidOrderEvent("Only schema_version=1 is supported")

    order_data = event.get("data")
    if not isinstance(order_data, dict):
        raise InvalidOrderEvent("The event must contain a data object")
    if not isinstance(order_data.get("order_id"), str) or not order_data["order_id"].strip():
        raise InvalidOrderEvent("data.order_id must be a non-empty string")

    # Stage 4.5 — Return only after every required check passes.
    return event


decoded_sample = decode_order_event(SAMPLE_ORDER_BYTES)
print("Decoded event ID:", decoded_sample["event_id"])


Decoded event ID: evt-offset-1001


### Code explanation — every meaningful line in the decoder

| Code | Meaning |
|---|---|
| `class InvalidOrderEvent(ValueError):` | Gives unsupported data a specific, recognizable error type while keeping normal `ValueError` behavior. |
| The class docstring | Defines which boundary failed and guarantees that bad data is not treated as processed. |
| `def decode_order_event(...) -> dict[str, Any]:` | Defines a function that accepts bytes or `None` and returns a dictionary only on success. `|` means “or.” |
| The function docstring | Documents input, output, failure behavior, and the poison-record tradeoff. |
| `if raw_value is None:` | Rejects a missing value because this order contract does not support tombstones/null records. |
| `raise InvalidOrderEvent(...)` | Stops immediately. The caller must not process or commit this record as successful. |
| `try:` | Starts conversion operations that can fail on untrusted input. |
| `raw_value.decode("utf-8")` | Converts record bytes into text. |
| `json.loads(...)` | Parses the text into a Python value. |
| `except (...) as error:` | Catches invalid byte encoding or invalid JSON and names the original cause. |
| `raise ... from error` | Reports a clear application error while retaining the technical cause for debugging. |
| `if not isinstance(event, dict):` | Rejects valid JSON that is not the object shape this handler expects. |
| The `event_type` and `schema_version` checks | Reject events whose meaning or shape this lesson does not support. |
| `order_data = event.get("data")` | Gives the nested business data a readable local name. |
| The `order_data` check | Ensures the nested value is a dictionary before reading its fields. |
| The `order_id` check | Rejects a missing, non-string, empty, or whitespace-only identifier. |
| `return event` | Returns only after every required condition passes. |
| `decoded_sample = ...` | Calls the decoder with the sample bytes. |
| The final `print(...)` | Proves that the decoded dictionary is usable. |


In [7]:
PROCESSED_EVENT_IDS: list[str] = []


def handle_order_created(event: dict[str, Any]) -> None:
    """Perform the lesson's visible business action after validation.

    Args:
        event: An event already accepted by ``decode_order_event``.

    Returns:
        Nothing. Returning normally means the demonstration action succeeded.

    Raises:
        KeyError: A required validated field is unexpectedly absent.

    Production tradeoff:
        This in-memory list disappears on restart and does not prevent duplicate
        effects. Production storage should use ``event_id`` idempotently.
    """
    # Stage 4.6 — Read the identity used to recognize this event.
    event_id = event["event_id"]

    # Stage 4.7 — Perform the demonstration business action.
    PROCESSED_EVENT_IDS.append(event_id)
    print(f"Business processing finished for {event_id}")


handle_order_created(decoded_sample)
print("Processed event IDs:", PROCESSED_EVENT_IDS)


Business processing finished for evt-offset-1001
Processed event IDs: ['evt-offset-1001']


### Code explanation — every meaningful line in the handler

| Code | Meaning |
|---|---|
| `PROCESSED_EVENT_IDS: list[str] = []` | Creates an in-memory list so the lesson can display completed work. It is not durable storage. |
| `def handle_order_created(...) -> None:` | Defines the business function. `None` means success is represented by returning normally. |
| The docstring | Defines its validated input, success rule, possible failure, and production limitation. |
| `event_id = event["event_id"]` | Reads the stable identity. Square brackets are used because validation is expected to guarantee the field. |
| `PROCESSED_EVENT_IDS.append(event_id)` | Performs the demonstration action by adding the identity to the list. |
| `print(f"...")` | Makes business completion visible; the `f` inserts `event_id`. |
| `handle_order_created(decoded_sample)` | Calls the handler with validated sample data. |
| The final `print(...)` | Shows the list state after that call. |

The polling loop will commit only if `handle_order_created()` returns normally. If it raises, the commit line is never reached.


In [8]:
def create_manual_consumer() -> Consumer:
    """Create a real consumer with automatic offset movement disabled.

    Args:
        None. The function uses ``MANUAL_COMMIT_CONFIG`` from Stage 3.

    Returns:
        A new, not-yet-subscribed Kafka consumer.

    Raises:
        RuntimeError: The optional Kafka client is not installed.
        KafkaException: The Kafka client rejects its configuration.

    Production tradeoff:
        Visible configuration is educational. Deployed credentials and endpoints
        should come from managed configuration rather than source code.
    """
    # Stage 4.8 — Fail with an installation instruction before using Kafka.
    if not KAFKA_LIBRARY_AVAILABLE or Consumer is None:
        raise RuntimeError("Install confluent-kafka before running the live lab")

    # Stage 4.9 — Construction alone does not subscribe, poll, or commit.
    return Consumer(MANUAL_COMMIT_CONFIG)


def commit_processed_message(consumer: Consumer, message: Message) -> None:
    """Synchronously commit the position after one processed message.

    Args:
        consumer: Active client that returned ``message``.
        message: Normal record whose business handler completed successfully.

    Returns:
        Nothing after Kafka confirms the synchronous commit.

    Raises:
        RuntimeError: No commit result is returned or a partition result fails.
        KafkaException: The commit request fails at the client/broker boundary.

    Production tradeoff:
        Waiting for every record's commit is easy to audit but adds a network
        round trip. Batching can improve speed but enlarges the replay window.
    """
    # Stage 4.10 — Commit message.offset() + 1 and wait for Kafka's result.
    committed_positions = consumer.commit(message=message, asynchronous=False)
    if not committed_positions:
        raise RuntimeError("Kafka returned no committed position")

    # Stage 4.11 — Inspect every partition result instead of assuming success.
    for position in committed_positions:
        if position.err is not None:
            raise RuntimeError(f"Kafka commit failed: {position.err}")


### Code explanation — every meaningful line in the consumer and commit helpers

| Code | Meaning |
|---|---|
| `def create_manual_consumer() -> Consumer:` | Defines one construction function and promises a consumer on success. |
| Its docstring | Explains that the function uses the visible manual configuration and may fail before any polling occurs. |
| The package check | Refuses a confusing call to a missing class and gives the installation action. |
| `return Consumer(MANUAL_COMMIT_CONFIG)` | Constructs the client with both automatic offset switches disabled. It has not subscribed yet. |
| `def commit_processed_message(...) -> None:` | Defines the only operation allowed to move the broker bookmark in the manual flow. |
| Its docstring | States the processed-message precondition, completion contract, failures, and per-record performance cost. |
| `consumer.commit(message=message, asynchronous=False)` | Requests the position after this message and blocks until the request succeeds or fails. For message offset `7`, it requests position `8`. |
| `if not committed_positions:` | Treats a missing commit result as failure rather than claiming success. |
| `for position in committed_positions:` | Checks every topic-partition result returned by the synchronous commit. |
| `if position.err is not None:` | Uses the commit API's per-partition error field to detect a failed position. |
| `raise RuntimeError(...)` | Stops with the actual partition error. The business handler may already have run, so replay remains possible. |

Why can a commit failure still lead to replay after business success? The business system and Kafka are separate. If the bookmark did not certainly move, a later consumer may receive the record again.


## Stage 4.12 — Read the complete algorithm before its code

1. Validate the notebook stopping limits.
2. Create a manual consumer.
3. Subscribe to the offset-lab topic.
4. Poll for one result.
5. Treat `None` as an ordinary empty wait.
6. Treat `message.error()` as a Kafka/client event, not business bytes.
7. Decode and validate a normal message.
8. Run the business handler.
9. Commit synchronously only after the handler succeeds.
10. Repeat until a notebook limit is reached.
11. Always close the consumer.


In [9]:
def consume_with_manual_commit(max_messages: int = 3, max_empty_polls: int = 10) -> int:
    """Consume orders with an explicit process-then-commit boundary.

    Args:
        max_messages: Stop after this many confirmed process-and-commit cycles.
        max_empty_polls: Stop after this many consecutive one-second empty polls.

    Returns:
        Number of records whose handler and synchronous commit both succeeded.

    Raises:
        ValueError: Either notebook stopping limit is not positive.
        InvalidOrderEvent: A normal record has unsupported data.
        KafkaException: Polling or committing reports a Kafka failure.
        Exception: The business handler fails; its original failure is preserved.

    Production tradeoff:
        Notebook limits prevent an endless cell. A service normally runs until a
        shutdown signal and adds retries, observability, and rebalance handling.
    """
    # Stage 4.13 — Reject limits that cannot describe a useful run.
    if max_messages <= 0 or max_empty_polls <= 0:
        raise ValueError("Both stopping limits must be positive")

    # Stage 4.14 — Create the client and counters before the loop.
    consumer = create_manual_consumer()
    completed_count = 0
    consecutive_empty_polls = 0

    try:
        # Stage 4.15 — Join the group and request work from one topic.
        consumer.subscribe([OFFSET_LAB_TOPIC])

        # Stage 4.16 — Continue while both notebook budgets remain.
        while completed_count < max_messages and consecutive_empty_polls < max_empty_polls:
            message = consumer.poll(1.0)

            # Stage 4.17 — No record in this wait means no work and no commit.
            if message is None:
                consecutive_empty_polls += 1
                print("No record arrived during this one-second poll")
                continue

            # Stage 4.18 — A Kafka error is not an order event.
            if message.error():
                raise KafkaException(message.error())

            consecutive_empty_polls = 0
            print(
                "Fetched record:",
                message.topic(),
                "partition", message.partition(),
                "offset", message.offset(),
            )

            # Stage 4.19 — Processing must finish before the bookmark moves.
            event = decode_order_event(message.value())
            handle_order_created(event)

            # Stage 4.20 — Commit and count only after business success.
            commit_processed_message(consumer, message)
            completed_count += 1
            print("Committed next position:", message.offset() + 1)
    finally:
        # Stage 4.21 — Release group membership on success or failure.
        consumer.close()
        print("Consumer closed")

    return completed_count


### Code explanation — every meaningful line in the complete manual loop

#### Contract and setup

| Code | Meaning |
|---|---|
| `def consume_with_manual_commit(...):` | Defines a bounded learning loop. It stops after three successful records or ten consecutive empty waits by default. |
| The docstring | Documents inputs, returned count, failure paths, and why production lifecycle control is larger. |
| The limit check | Rejects zero or negative limits that would make the teaching run unclear. |
| `consumer = create_manual_consumer()` | Creates a client whose automatic store and commit switches are off. |
| `completed_count = 0` | Counts only records whose business handler and synchronous commit both succeeded. |
| `consecutive_empty_polls = 0` | Tracks an idle streak so an empty notebook topic does not wait forever. |

#### Subscribe and poll

| Code | Meaning |
|---|---|
| `try:` | Begins a protected region whose `finally` cleanup runs on normal exit or failure. |
| `consumer.subscribe([OFFSET_LAB_TOPIC])` | Requests group work from a list containing the one lab topic. |
| The `while` condition | Repeats only while neither notebook limit has been reached. |
| `message = consumer.poll(1.0)` | Waits up to one second for a record/event and stores the returned answer. |
| `if message is None:` | Selects the normal empty-wait path. |
| `consecutive_empty_polls += 1` | Adds one to the idle streak. |
| The empty-poll `print(...)` | Makes the reason for waiting visible while learning. |
| `continue` | Jumps to the next poll without decoding, processing, or committing. |

#### Handle a real result

| Code | Meaning |
|---|---|
| `if message.error():` | Detects that Kafka returned an error/event instead of normal business bytes. |
| `raise KafkaException(...)` | Stops and preserves Kafka's error details. No commit happens on this path. |
| `consecutive_empty_polls = 0` | A real record breaks the idle streak. |
| The multi-line metadata `print(...)` | Displays topic, partition, and record offset. Parentheses only let the one call span readable lines. |
| `event = decode_order_event(message.value())` | Converts and validates untrusted record bytes. Failure stops before business work and commit. |
| `handle_order_created(event)` | Performs the business action. Failure stops before commit. |
| `commit_processed_message(...)` | Requests and verifies the next broker position only after both previous lines succeed. |
| `completed_count += 1` | Counts success only after Kafka's synchronous commit helper returns successfully. |
| `message.offset() + 1` | Displays the next reading position that was requested. |

#### Cleanup and result

| Code | Meaning |
|---|---|
| `finally:` | Declares cleanup that runs for success, stopping limits, or exceptions. |
| `consumer.close()` | Stops consumption, leaves the group, and releases client resources. Because auto-commit is disabled here, close is not used to save unfinished work. |
| `print("Consumer closed")` | Makes cleanup visible. |
| `return completed_count` | Returns the completed process-and-commit count when no exception escaped. |


---
# Stage 5.0 — Optional live Kafka experiment

The next helper sends one known event so the manual consumer has something to read. It exists only to make this notebook self-contained.

To run it:

1. Start the learning broker.
2. Install `confluent-kafka`.
3. Change `RUN_LIVE_KAFKA` to `True` in Stage 0.2.
4. Restart the kernel and run all cells.

The lab assumes the learning broker permits automatic topic creation. If your broker disables that feature, create `beginner_offset_commit_lab` before running.


In [10]:
def publish_offset_lab_event() -> None:
    """Send the sample event so the live manual consumer can read it.

    Args:
        None. Uses the visible broker, topic, and sample bytes.

    Returns:
        Nothing after Kafka acknowledges the one record.

    Raises:
        RuntimeError: The Kafka package is missing or delivery is incomplete.

    Production tradeoff:
        This one-record helper waits for all delivery work for clarity. Production
        producer lifecycle and batching belong in the producer lessons.
    """
    # Stage 5.1 — Check the optional library before constructing a producer.
    if not KAFKA_LIBRARY_AVAILABLE or Producer is None:
        raise RuntimeError("Install confluent-kafka before running the live lab")

    # Stage 5.2 — Create the producer and containers for its callback result.
    producer = Producer({"bootstrap.servers": BOOTSTRAP_SERVERS, "acks": "all"})
    acknowledged_offsets: list[int] = []
    delivery_failures: list[str] = []

    def record_delivery(error: object | None, message: Message) -> None:
        """Capture the asynchronous delivery result for this one-record lab."""
        if error is not None:
            delivery_failures.append(str(error))
            return
        acknowledged_offsets.append(message.offset())

    # Stage 5.3 — Queue one known record for Kafka delivery.
    producer.produce(
        topic=OFFSET_LAB_TOPIC,
        key=b"order-1001",
        value=SAMPLE_ORDER_BYTES,
        on_delivery=record_delivery,
    )

    # Stage 5.4 — Wait for the callback and reject an uncertain result.
    remaining_messages = producer.flush(10.0)
    if remaining_messages != 0 or delivery_failures or not acknowledged_offsets:
        raise RuntimeError(
            "Sample delivery was incomplete: "
            f"remaining={remaining_messages}, failures={delivery_failures}"
        )
    print("Producer wrote sample record at offset:", acknowledged_offsets[0])


### Code explanation — every meaningful line in the one-record producer helper

| Code | Meaning |
|---|---|
| `def publish_offset_lab_event() -> None:` | Defines a self-contained helper that returns normally only after one acknowledged send. |
| Its docstring | States its existing inputs, completion rule, failure behavior, and limited teaching purpose. |
| The Kafka package check | Produces an installation instruction instead of a vague missing-class error. |
| `Producer({...})` | Creates a producer for the same broker. `acks="all"` requests acknowledgement from all configured in-sync replicas; the learning broker has one broker. |
| `acknowledged_offsets = []` | Collects the broker offset reported after successful delivery. |
| `delivery_failures = []` | Collects readable error messages from failed delivery. |
| `def record_delivery(...)` | Defines the callback served when the producer learns the final delivery result. |
| The callback docstring | States its one purpose. |
| `if error is not None:` | Selects failed delivery. |
| `delivery_failures.append(str(error))` | Saves readable error information. |
| `return` | Ends the failed callback path so it is not counted as success. |
| `acknowledged_offsets.append(message.offset())` | Saves the Kafka-assigned offset on success. |
| `producer.produce(...)` | Queues one record with an explicit topic, key, byte value, and callback. Queueing is not yet broker acknowledgement. |
| `producer.flush(10.0)` | Waits up to ten seconds for queued delivery work and returns the unfinished count. |
| The three-part `if` | Rejects unfinished work, a reported failure, or a missing acknowledgement. |
| `raise RuntimeError(...)` | Reports pending work and captured failures. Adjacent strings inside parentheses form one message. |
| The final `print(...)` | Displays the record offset assigned by Kafka. |


In [ ]:
# Stage 5.5 — Network work occurs only behind the explicit safety switch.
if RUN_LIVE_KAFKA:
    publish_offset_lab_event()
    completed = consume_with_manual_commit(max_messages=1, max_empty_polls=10)
    print("Live process-and-commit count:", completed)
else:
    print("Live Kafka skipped. Set RUN_LIVE_KAFKA=True when the broker is ready.")


%3|1792639345.934|FAIL|rdkafka#producer-1| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv4#127.0.0.1:9092 failed: Connection refused (after 2ms in state CONNECT)
%3|1792639345.984|FAIL|rdkafka#producer-1| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv4#127.0.0.1:9092 failed: Connection refused (after 0ms in state CONNECT)
%3|1792639346.037|FAIL|rdkafka#producer-1| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv4#127.0.0.1:9092 failed: Connection refused (after 1ms in state CONNECT)
%3|1792639346.250|FAIL|rdkafka#producer-1| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv4#127.0.0.1:9092 failed: Connection refused (after 0ms in state CONNECT)
%3|1792639346.708|FAIL|rdkafka#producer-1| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv4#127.0.0.1:9092 failed: Connection refused (after 0ms in state CONNECT)
%3|1792639346.933|FAIL|rdkafka#producer-1| [thrd:localh

RuntimeError: Sample delivery was incomplete: remaining=1, failures=[]

%3|1792639356.942|FAIL|rdkafka#producer-1| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv4#127.0.0.1:9092 failed: Connection refused (after 1ms in state CONNECT)
%3|1792639356.993|FAIL|rdkafka#producer-1| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv4#127.0.0.1:9092 failed: Connection refused (after 1ms in state CONNECT)
%3|1792639357.373|FAIL|rdkafka#producer-1| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv4#127.0.0.1:9092 failed: Connection refused (after 1ms in state CONNECT)
%3|1792639357.942|FAIL|rdkafka#producer-1| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv4#127.0.0.1:9092 failed: Connection refused (after 0ms in state CONNECT)
%3|1792639357.993|FAIL|rdkafka#producer-1| [thrd:localhost:9092/bootstrap]: localhost:9092/bootstrap: Connect to ipv4#127.0.0.1:9092 failed: Connection refused (after 0ms in state CONNECT)
%3|1792639358.373|FAIL|rdkafka#producer-1| [thrd:localh

### Code explanation — every line in Stage 5.5

| Code | Meaning |
|---|---|
| The Stage 5.5 comment | Marks the only path that performs network work. |
| `if RUN_LIVE_KAFKA:` | Runs the indented live branch only after you deliberately enable it. |
| `publish_offset_lab_event()` | Sends and waits for acknowledgement of one known record. |
| `consume_with_manual_commit(...)` | Reads at most one successfully committed record and stops after ten consecutive empty polls if none arrives. |
| The success `print(...)` | Shows how many records completed both business work and synchronous commit. |
| `else:` | Selects the safe offline path while the switch remains `False`. |
| The skip `print(...)` | Explains why no Kafka connection occurred and what to change deliberately. |

Because `MANUAL_GROUP_ID` is stable, later runs resume from that group's last committed position. The newly published record appears after the previous position and can be consumed next.


---
# Stage 6.0 — Understand the middle ground after manual commit

Mode C uses:

```python
"enable.auto.offset.store": False
"enable.auto.commit": True
```

The application decides when an offset becomes eligible, but the client decides when to send stored progress to Kafka.


In [ ]:
def process_then_store(consumer: Consumer, message: Message) -> None:
    """Process one record, then make its next position eligible for auto-commit.

    Args:
        consumer: Client configured with ``MANUAL_STORE_CONFIG``.
        message: One normal Kafka record with no message-level error.

    Returns:
        Nothing after processing succeeds and the offset is stored locally.

    Raises:
        InvalidOrderEvent: The record value is unsupported.
        Exception: Business processing or local offset storage fails.

    Production tradeoff:
        This avoids a synchronous network round trip per record, but the stored
        position is not yet a confirmed Kafka commit and can be lost on a crash.
    """
    # Stage 6.1 — Finish application work first.
    event = decode_order_event(message.value())
    handle_order_created(event)

    # Stage 6.2 — Mark the next position locally; background commit happens later.
    consumer.store_offsets(message=message)


### Code explanation — every meaningful line in the middle-ground helper

| Code | Meaning |
|---|---|
| `def process_then_store(...) -> None:` | Defines one-record behavior for a client using the manual-store configuration. |
| Its docstring | States that success means processing plus **local storage**, not a confirmed broker commit. |
| `event = decode_order_event(message.value())` | Converts and validates the record bytes. Failure prevents processing and local storage. |
| `handle_order_created(event)` | Performs business work. Failure prevents local storage. |
| `consumer.store_offsets(message=message)` | Stores `message.offset() + 1` inside the client. It does not synchronously write that position to Kafka. |

Possible timeline:

```text
process offset 7 successfully
        |
        v
store next position 8 locally
        |
        |  <--- crash here: local state disappears
        v
background auto-commit writes 8 to Kafka
```

If the application crashes before the background commit, record `7` can replay. This mode avoids committing unfinished work while reducing per-record commit waiting, but commit confirmation and failure timing are less direct than the synchronous baseline.


---
# Stage 7.0 — Synchronous, asynchronous, and stored are not the same

| API call | What has happened when the call returns? |
|---|---|
| `commit(message=msg, asynchronous=False)` | The call waited for a broker result. Check exceptions and every returned partition error. |
| `commit(message=msg, asynchronous=True)` | The request was queued. A successful broker commit is not yet confirmed to this line of code. |
| `store_offsets(message=msg)` | The next position was stored inside the client. No broker commit is confirmed. |

Asynchronous commit may improve throughput, but the application must handle commit callbacks, failures, and commits completing in a different order than they were requested. The callbacks are served as the consumer polls.

For a first implementation, synchronous per-record commit is slower but easier to reason about:

```text
handler success → wait for commit result → move to next record
```

Notebook 9 can introduce batching after this safety boundary is understood.


## Stage 7.1 — What happens in each failure case?

| Failure moment | Broker bookmark | Likely restart behavior |
|---|---|---|
| Before `poll()` returns a record | Unchanged | Read from the previous committed position. |
| After poll, before processing | Unchanged in manual mode | Record can be delivered again. |
| During failed processing | Unchanged in manual mode | Record can be delivered again. |
| After processing, before commit | Unchanged | Business work may repeat after replay. |
| Commit confirmed after processing | Advanced to next position | Group normally continues after the record. |
| Commit response times out | Uncertain | Broker may or may not have committed; design for replay. |

A timeout is **ambiguous**. “My application did not receive confirmation” is not always the same as “Kafka definitely did not save the commit.” Do not solve that uncertainty by assuming the business action cannot repeat.


## Stage 7.2 — Production limits translated into ordinary questions

| Production term | Question to ask |
|---|---|
| **Idempotency** | If this event runs again, will the final business result still be correct? |
| **Poison record** | Is one permanently invalid record repeatedly blocking this partition? |
| **Rebalance** | Did Kafka move this partition to another member of the consumer group? |
| **Commit latency** | How long does Kafka take to confirm our saved progress? |
| **Consumer lag** | How far behind the newest partition position is this group? |
| **Max poll interval** | Can our handler finish before Kafka decides this consumer stopped polling for too long? |

A production service also needs:

- an explicit retry and bad-record quarantine policy;
- durable idempotency using a database constraint, inbox, or external API key;
- metrics for handler failures, commit failures, processing time, and lag;
- graceful shutdown that stops intake and commits only known-safe progress;
- rebalance-aware ownership rules before introducing multiple consumers;
- authentication, encryption, and secret management;
- integration checks against a disposable Kafka broker.

These requirements do not change the beginner flow. They protect each boundary when failures, restarts, and multiple processes are introduced.


---
# Final comparison

| Mode | Who marks an offset eligible? | Who commits to Kafka? | Main tradeoff |
|---|---|---|---|
| Auto-store + auto-commit | Client, when records are delivered | Client, periodically | Very little code, but progress timing is not tied to business success. |
| Manual commit after processing | Application, by naming the processed message | Application, explicitly | Clear safety boundary, but synchronous per-record commits cost throughput. |
| Manual store + auto-commit | Application, after processing | Client, periodically | Avoids early storage and per-record waiting, but broker commit is delayed. |

## Five rules to remember

1. After processing record offset `7`, the next committed position is `8`.
2. Fetched, processed, locally stored, and broker-committed are four different moments.
3. Commit before processing risks skipped work.
4. Process before commit risks replay, so business effects must tolerate duplicates.
5. A Kafka commit saves consumer-group progress; it does not prove an external business transaction happened exactly once.

The one sentence to say out loud is:

> **Manual commit means my application moves Kafka's bookmark only after my business work succeeds. If a crash happens before that bookmark moves, I accept replay and make the business work duplicate-safe.**

## Next notebook

Notebook 8 explains what changes when several consumers share a group and Kafka redistributes partition ownership.

Authoritative references:

- [Confluent Python client overview — delivery guarantees and offset storage](https://docs.confluent.io/kafka-clients/python/current/overview.html)
- [Confluent Python API — `Consumer.commit()` and `store_offsets()`](https://docs.confluent.io/platform/current/clients/confluent-kafka-python/html/index.html)
